In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
from datasets import load_from_disk

data_dir = "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-c"
ds = load_from_disk(data_dir)

print(ds)
print("Train columns:", ds["train"].column_names)
print("Validation columns:", ds["validation"].column_names)
print("Sample:", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 900000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 200000
    })
})
Train columns: ['label', 'input_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'attention_mask']
Sample: {'label': 1, 'input_ids': [0, 4308, 10427, 130, 399, 1393, 1086, 130, 1393, 5559, 425, 1029, 464, 14073, 1177, 317, 4308, 399, 1393, 9381, 425, 1029, 464, 1313, 131, 191, 2243, 1313, 1177, 317, 4308, 7087, 3663, 1029, 19967, 4461, 3663, 1177, 317, 317, 1110, 46802, 2542, 841, 385, 15031, 4395, 130, 2297, 130, 611, 507, 130, 2308, 157, 507, 130, 7255, 157, 507, 2802, 771, 399, 317, 339, 925, 399, 422, 425, 385, 1393, 9381, 523, 317, 339, 925, 626, 4273, 697, 130, 827, 3838, 697, 179, 385, 1393, 1086, 126, 37028, 853, 4395, 132, 1338, 388, 317, 317, 339, 1393, 5559, 7590, 771, 399, 317, 377, 827, 3838, 697, 126, 37028, 85

In [3]:
!pip install -U transformers datasets evaluate accelerate pyarrow scikit-learn

In [4]:
import os

data_dir = "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-c"

print("Danh sách file:")
for f in os.listdir(data_dir):
    print("-", f)

Danh sách file:
- dataset_dict.json
- train
- validation


In [5]:
new_train_code = r'''
import os
import json
import argparse
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
import evaluate
import torch


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model_name", type=str, default="microsoft/unixcoder-base")
    parser.add_argument("--resume_checkpoint", type=str, default=None)
    parser.add_argument("--num_train_epochs", type=int, default=3)
    parser.add_argument("--per_device_train_batch_size", type=int, default=4)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=4)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=4)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--logging_steps", type=int, default=5000)
    parser.add_argument("--save_total_limit", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--bf16", action="store_true")
    return parser.parse_args()


def infer_label_info(train_ds, label_col):
    values = train_ds[label_col]
    unique_labels = sorted(list(set(values)))
    label2id = {str(int(v)): int(v) for v in unique_labels}
    id2label = {int(v): str(int(v)) for v in unique_labels}
    return len(unique_labels), label2id, id2label


def encode_labels(example, label_col, label2id):
    example["labels"] = int(label2id[str(int(example[label_col]))])
    return example


class TokenizedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        import torch
        labels = [f["labels"] for f in features]
        features_no_labels = [{k: v for k, v in f.items() if k != "labels"} for f in features]

        batch = self.tokenizer.pad(
            features_no_labels,
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


class OneLineProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        parts = []
        if "loss" in logs:
            parts.append(f"loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            parts.append(f"lr={logs['learning_rate']:.2e}")
        if "epoch" in logs:
            parts.append(f"epoch={logs['epoch']:.4f}")
        parts.append(f"step={state.global_step}")
        print("\r[TRAIN] " + " | ".join(parts), end="", flush=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        print()
        if metrics:
            parts = [f"{k}={v:.4f}" for k, v in metrics.items() if isinstance(v, (int, float))]
            print("[EVAL] " + " | ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print()


def build_compute_metrics():
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
            "f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        }

    return compute_metrics


def main():
    args = parse_args()
    os.makedirs(args.output_dir, exist_ok=True)

    if args.fp16 and not torch.cuda.is_available():
        raise RuntimeError("Resume fp16 chỉ nên chạy khi đang bật GPU trong Colab.")

    dataset_dict = load_from_disk(args.data_dir)
    label_col = "label"

    num_labels, label2id, id2label = infer_label_info(dataset_dict["train"], label_col)

    processed = {}
    for split in dataset_dict.keys():
        ds = dataset_dict[split].map(
            lambda x: encode_labels(x, label_col, label2id),
            load_from_cache_file=True,
            desc=f"encode_{split}"
        )
        keep_cols = ["input_ids", "attention_mask", "labels"]
        remove_cols = [c for c in ds.column_names if c not in keep_cols]
        processed[split] = ds.remove_columns(remove_cols)

    train_dataset = processed["train"]
    eval_dataset = processed["validation"]

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        args.model_name,
        num_labels=num_labels,
        label2id=label2id,
        id2label=id2label,
    )

    resume_checkpoint = args.resume_checkpoint
    if resume_checkpoint is not None and not os.path.isdir(resume_checkpoint):
        raise ValueError(f"Không tìm thấy checkpoint: {resume_checkpoint}")

    model_source = resume_checkpoint if resume_checkpoint is not None else args.model_name
    print("Load model từ:", model_source)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_source,
        config=config,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.num_train_epochs,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        logging_steps=args.logging_steps,
        save_steps=20000,
        save_total_limit=2,
        fp16=args.fp16,
        bf16=args.bf16,
        seed=args.seed,
        report_to="none",
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=TokenizedDataCollator(tokenizer),
        compute_metrics=build_compute_metrics(),
        callbacks=[OneLineProgressCallback()],
    )

    if resume_checkpoint is not None:
        print("Resume training state từ:", resume_checkpoint)
        trainer.train(resume_from_checkpoint=resume_checkpoint)
    else:
        trainer.train()

    best_model_dir = os.path.join(args.output_dir, "best_model")
    os.makedirs(best_model_dir, exist_ok=True)

    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)

    with open(os.path.join(best_model_dir, "label2id.json"), "w", encoding="utf-8") as f:
        json.dump(label2id, f, ensure_ascii=False, indent=2)

    with open(os.path.join(best_model_dir, "id2label.json"), "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

    meta = {
        "model_name": args.model_name,
        "task_name": os.path.basename(args.data_dir),
        "num_labels": num_labels,
        "label_column": label_col,
        "train_size": len(train_dataset),
        "validation_size": len(eval_dataset),
        "best_metric_name": args.metric_for_best_model,
        "validation_metrics": eval_metrics,
    }

    with open(os.path.join(best_model_dir, "model_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    metrics = trainer.evaluate()
    print("[FINAL]", metrics)


if __name__ == "__main__":
    main()
'''
with open("train_unixcoder.py", "w", encoding="utf-8") as f:
    f.write(new_train_code)

print("Đã cập nhật train_unixcoder.py")

Đã cập nhật train_unixcoder.py


In [6]:
!python train_unixcoder.py \
  --data_dir "/content/drive/MyDrive/SemEval2026/sem-eval-2026-task13-subtask-c" \
  --output_dir "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c" \
  --resume_checkpoint "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/checkpoint-160000" \
  --num_train_epochs 3 \
  --per_device_train_batch_size 4 \
  --per_device_eval_batch_size 4 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-5 \
  --weight_decay 0.01 \
  --logging_steps 5000 \
  --fp16

config.json: 100% 691/691 [00:00<00:00, 4.08MB/s]
tokenizer_config.json: 1.11kB [00:00, 2.69MB/s]
vocab.json: 938kB [00:00, 26.5MB/s]
merges.txt: 444kB [00:00, 99.4MB/s]
special_tokens_map.json: 100% 772/772 [00:00<00:00, 4.09MB/s]
Load model từ: /content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/checkpoint-140000
Loading weights: 100% 201/201 [00:02<00:00, 76.61it/s]
Resume training state từ: /content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/checkpoint-140000
[transformers] There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer

In [7]:
output_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/best_model"

print("Các file trong best_model:")
for f in os.listdir(output_dir):
    print("-", f)

Các file trong best_model:
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json
- training_args.bin
- label2id.json
- id2label.json


In [8]:
from transformers import AutoTokenizer

model_name = "microsoft/unixcoder-base"
best_model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(best_model_dir)

print("Đã lưu lại tokenizer đầy đủ")
print(os.listdir(best_model_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Đã lưu lại tokenizer đầy đủ
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'label2id.json', 'id2label.json']


In [9]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_dir = "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_c/best_model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with open(f"{model_dir}/id2label.json", "r", encoding="utf-8") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

code_text = """
def add(a, b):
    return a + b
"""

inputs = tokenizer(
    code_text,
    truncation=True,
    padding=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

print({
    "predicted_label_id": pred_id,
    "predicted_label": id2label[pred_id],
    "confidence": float(probs[pred_id])
})

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'predicted_label_id': 1, 'predicted_label': '1', 'confidence': 0.6604852676391602}


In [10]:

  #--resume_checkpoint "/content/drive/MyDrive/SemEval2026/outputs/unixcoder_subtask_a/checkpoint-36000" \